# Stroke Log-Signature + MLP (Few-shot friendly)

Optional baseline using path log-signatures. Install iisignature locally (CPU). If not available, the notebook will skip.

In [ ]:
try:
    import iisignature as _sig
    HAVE_SIG=True
except Exception as e:
    HAVE_SIG=False; print('iisignature not available; install with: pip install iisignature')


In [ ]:
from pathlib import Path; import sys, numpy as np, torch
from torch import nn; from torch.utils.data import DataLoader
SRC=(Path('..')/'src').resolve(); sys.path.insert(0,str(SRC))
from stroke_recognizer.data import build_dataloaders, DEFAULT_LABELS_HEBREW
from stroke_recognizer.transforms import remove_duplicate_points, median_filter, normalize_center_scale, resample_strokes
DATA_ROOT=str((Path('..')/'data').resolve()); LABELS=DEFAULT_LABELS_HEBREW
N_POINTS=96; BATCH=64; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
if not HAVE_SIG: raise SystemExit('iisignature not installed; skipping notebook')
train_loader, val_loader, _ = build_dataloaders(root=DATA_ROOT, label_names=LABELS, n_points=N_POINTS, batch_size=BATCH)
len(train_loader.dataset), len(val_loader.dataset)


In [ ]:
def to_signature(batch_x, depth=2):
    # batch_x: (B,T,9) -> use xy + pen_up channel
    xs=[]
    for x in batch_x.cpu().numpy():
        path = np.stack([x[:,0], x[:,1], x[:,-1]], axis=1)
        xs.append(_sig.logsig(path, depth))
    arr=np.stack(xs,axis=0)
    return torch.from_numpy(arr).float()

# Probe one sample to get feature dimension
x0,_=next(iter(train_loader)); feat_dim=to_signature(x0[:1]).shape[1]; feat_dim


In [ ]:
class SigMLP(nn.Module):
    def __init__(self, d_in, num_classes):
        super().__init__(); self.net=nn.Sequential(nn.Linear(d_in,256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256,num_classes))
    def forward(self, feats): return self.net(feats)

model=SigMLP(feat_dim, len(LABELS)).to(device)
crit=nn.CrossEntropyLoss(label_smoothing=0.05)
opt=torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
def top1(logits,y): return (logits.argmax(1)==y).float().mean().item()


In [ ]:
hist={'val_top1':[]}
for epoch in range(1,31):
    model.train()
    for x,y in train_loader:
        feats=to_signature(x).to(device); y=y.to(device)
        opt.zero_grad(set_to_none=True); logits=model(feats); loss=crit(logits,y); loss.backward(); opt.step()
    # val
    model.eval(); acc=0.0; n=0
    with torch.no_grad():
        for x,y in val_loader:
            feats=to_signature(x).to(device); y=y.to(device)
            logits=model(feats); acc+=top1(logits,y)*x.size(0); n+=x.size(0)
    acc/=max(1,n); hist['val_top1'].append(acc); print(f'Epoch {epoch:02d} val_top1={acc:.3f}')
